# Fashion-MNIST Classifier

Built incrementally, following Chapter 10 of Géron's *Hands-On Machine
Learning with Scikit-Learn and PyTorch* (2025): load the data, create
DataLoaders, define the model, train it, make predictions, plot accuracy,
and tune hyperparameters with Optuna.

## 1. Load and split the dataset

In [ ]:
import torch
from torch.utils.data import random_split
from torchvision import datasets
from torchvision.transforms import ToTensor

SEED = 42
N_VALID = 5_000

torch.manual_seed(SEED)

full_train_dataset = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=ToTensor(),
)

test_dataset = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=ToTensor(),
)

n_train = len(full_train_dataset) - N_VALID
generator = torch.Generator().manual_seed(SEED)
train_dataset, valid_dataset = random_split(
    full_train_dataset, [n_train, N_VALID], generator=generator
)

print(f"Training set size: {len(train_dataset)}")
print(f"Validation set size: {len(valid_dataset)}")
print(f"Test set size: {len(test_dataset)}")

image, label = train_dataset[0]
print(f"Image shape: {image.shape}, dtype: {image.dtype}")
print(f"Pixel value range: [{image.min():.4f}, {image.max():.4f}]")
print(f"Label: {label}")


## 2. Create DataLoaders

In [ ]:
from torch.utils.data import DataLoader

BATCH_SIZE = 32

CLASS_NAMES = [
    "T-shirt/top",
    "Trouser",
    "Pullover",
    "Dress",
    "Coat",
    "Sandal",
    "Shirt",
    "Sneaker",
    "Bag",
    "Ankle boot",
]

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

image, label = train_dataset[0]
print(f"Image shape: {image.shape}")
print(f"Image dtype: {image.dtype}")
print(f"Class name: {CLASS_NAMES[label]}")


## 3. Define the model

In [ ]:
from torch import nn

torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


class FashionClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.hidden1 = nn.Linear(28 * 28, 300)
        self.hidden2 = nn.Linear(300, 100)
        self.output = nn.Linear(100, 10)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.flatten(x)
        x = self.relu(self.hidden1(x))
        x = self.relu(self.hidden2(x))
        return self.output(x)


model = FashionClassifier().to(device)
loss_fn = nn.CrossEntropyLoss()

print(f"Device: {device}")
print(model)


## 4. Train the model

In [ ]:
from torch import optim
from torchmetrics import Accuracy

N_EPOCHS = 20
LEARNING_RATE = 0.1

optimizer = optim.SGD(model.parameters(), lr=LEARNING_RATE)


def train(model, train_loader, valid_loader, loss_fn, optimizer, n_epochs, device):
    history = {
        "loss": [],
        "train_accuracy": [],
        "val_accuracy": [],
    }

    train_accuracy = Accuracy(task="multiclass", num_classes=10).to(device)
    val_accuracy = Accuracy(task="multiclass", num_classes=10).to(device)

    for epoch in range(n_epochs):
        model.train()
        train_accuracy.reset()
        running_loss = 0.0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = loss_fn(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)
            train_accuracy.update(outputs, labels)

        epoch_loss = running_loss / len(train_loader.dataset)
        epoch_train_accuracy = train_accuracy.compute().item()

        model.eval()
        val_accuracy.reset()
        with torch.no_grad():
            for images, labels in valid_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                val_accuracy.update(outputs, labels)

        epoch_val_accuracy = val_accuracy.compute().item()

        history["loss"].append(epoch_loss)
        history["train_accuracy"].append(epoch_train_accuracy)
        history["val_accuracy"].append(epoch_val_accuracy)

        print(
            f"Epoch {epoch + 1}/{n_epochs} - "
            f"loss: {epoch_loss:.4f} - "
            f"train_accuracy: {epoch_train_accuracy:.4f} - "
            f"val_accuracy: {epoch_val_accuracy:.4f}"
        )

    return history


history = train(model, train_loader, valid_loader, loss_fn, optimizer, N_EPOCHS, device)


## 5. Make predictions on validation samples

In [ ]:
N_SAMPLES = 3
TOP_K = 4

n_params = sum(p.numel() for p in model.parameters())
print(f"Total number of parameters: {n_params:,}")

model.eval()
images = torch.stack([valid_dataset[i][0] for i in range(N_SAMPLES)]).to(device)
labels = [valid_dataset[i][1] for i in range(N_SAMPLES)]

with torch.no_grad():
    logits = model(images)
    probabilities = torch.softmax(logits, dim=1)
    predictions = torch.argmax(probabilities, dim=1)

for i in range(N_SAMPLES):
    predicted_label = predictions[i].item()
    actual_label = labels[i]

    print(f"\nSample {i + 1}")
    print(f"  Predicted: {CLASS_NAMES[predicted_label]}")
    print(f"  Actual:    {CLASS_NAMES[actual_label]}")

    print("  Probability for each class:")
    for class_idx, class_name in enumerate(CLASS_NAMES):
        print(f"    {class_name:<15} {probabilities[i, class_idx].item():.4f}")

    top_probs, top_classes = torch.topk(probabilities[i], TOP_K)
    print(f"  Top {TOP_K} most likely classes:")
    for prob, class_idx in zip(top_probs, top_classes):
        print(f"    {CLASS_NAMES[class_idx.item()]:<15} {prob.item():.4f}")


## 6. Plot training accuracy

In [ ]:
import matplotlib.pyplot as plt

epochs = range(1, N_EPOCHS + 1)
plt.plot(epochs, history["train_accuracy"], marker="o", label="Training accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training Accuracy over Epochs")
plt.xticks(epochs)
plt.legend()
plt.grid(True)
plt.savefig("training_accuracy.png")
plt.show()


## 7. Tune hyperparameters with Optuna

Search the learning rate (1e-5 to 1e-1, log scale) and the number of neurons
in the hidden layers (20 to 300, shared by both layers). Each trial trains a
freshly built model for 10 epochs and is scored by its best validation
accuracy.

In [ ]:
import optuna

OPTUNA_SEED = 42
N_TRIALS_BASIC = 5
N_EPOCHS_BASIC = 10


def build_tunable_model(n_hidden):
    return nn.Sequential(
        nn.Flatten(),
        nn.Linear(28 * 28, n_hidden),
        nn.ReLU(),
        nn.Linear(n_hidden, n_hidden),
        nn.ReLU(),
        nn.Linear(n_hidden, 10),
    ).to(device)


def objective_basic(trial):
    lr = trial.suggest_float("lr", 1e-5, 1e-1, log=True)
    n_hidden = trial.suggest_int("n_hidden", 20, 300)

    torch.manual_seed(OPTUNA_SEED)
    tuned_model = build_tunable_model(n_hidden)
    tuned_loss_fn = nn.CrossEntropyLoss()
    tuned_optimizer = optim.SGD(tuned_model.parameters(), lr=lr)

    trial_history = train(
        tuned_model,
        train_loader,
        valid_loader,
        tuned_loss_fn,
        tuned_optimizer,
        N_EPOCHS_BASIC,
        device,
    )
    return max(trial_history["val_accuracy"])


sampler = optuna.samplers.TPESampler(seed=OPTUNA_SEED)
study = optuna.create_study(direction="maximize", sampler=sampler)
study.optimize(objective_basic, n_trials=N_TRIALS_BASIC)

print("\nBest parameters:", study.best_params)
print("Best validation accuracy:", study.best_value)


## 8. Optuna search with early pruning

Improves on the previous search by training one epoch at a time, reporting
validation accuracy to Optuna after every epoch, and using a median pruner
to stop unpromising trials early. The data loaders and device are passed
explicitly into the objective function instead of relying on globals.

In [ ]:
from functools import partial

N_TRIALS_PRUNED = 20
N_EPOCHS_PRUNED = 10


def build_model_for_pruning(n_hidden):
    return nn.Sequential(
        nn.Flatten(),
        nn.Linear(28 * 28, n_hidden),
        nn.ReLU(),
        nn.Linear(n_hidden, n_hidden),
        nn.ReLU(),
        nn.Linear(n_hidden, 10),
    ).to(device)


def train_one_epoch(model, loader, loss_fn, optimizer, device):
    model.train()
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = loss_fn(outputs, labels)
        loss.backward()
        optimizer.step()


def evaluate_accuracy(model, loader, device):
    model.eval()
    accuracy = Accuracy(task="multiclass", num_classes=10).to(device)
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            accuracy.update(outputs, labels)
    return accuracy.compute().item()


def objective_pruning(trial, train_loader, valid_loader, device):
    lr = trial.suggest_float("lr", 1e-5, 1e-1, log=True)
    n_hidden = trial.suggest_int("n_hidden", 20, 300)

    torch.manual_seed(OPTUNA_SEED)
    tuned_model = build_model_for_pruning(n_hidden)
    tuned_loss_fn = nn.CrossEntropyLoss()
    tuned_optimizer = optim.SGD(tuned_model.parameters(), lr=lr)

    best_val_accuracy = 0.0
    for epoch in range(N_EPOCHS_PRUNED):
        train_one_epoch(tuned_model, train_loader, tuned_loss_fn, tuned_optimizer, device)
        val_accuracy = evaluate_accuracy(tuned_model, valid_loader, device)
        best_val_accuracy = max(best_val_accuracy, val_accuracy)

        trial.report(val_accuracy, epoch)
        if trial.should_prune():
            raise optuna.TrialPruned()

    return best_val_accuracy


pruning_sampler = optuna.samplers.TPESampler(seed=OPTUNA_SEED)
pruner = optuna.pruners.MedianPruner()
pruned_study = optuna.create_study(direction="maximize", sampler=pruning_sampler, pruner=pruner)
pruned_study.optimize(
    partial(objective_pruning, train_loader=train_loader, valid_loader=valid_loader, device=device),
    n_trials=N_TRIALS_PRUNED,
)

print("\nBest parameters:", pruned_study.best_params)
print("Best validation accuracy:", pruned_study.best_value)
